<a href="https://colab.research.google.com/github/fabiopauli/Qwen3.5-colab/blob/main/LTX2_Server_L4_GPU_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎥 LTX-2 Video Generation Server (L4 GPU)

Run **LTX-2.3** (Lightricks' DiT-based audio-video foundation model) on a **Google Colab L4 GPU** (22 GB VRAM)
and expose it as a **FastAPI server** via **ngrok**.

Uses the **DistilledPipeline** with **FP8 quantization** for fast inference that fits in 22 GB VRAM.

---

## ⚙️ Prerequisites — Colab Secrets

Before running, add these secrets in the Colab sidebar (🔑 icon → "Secrets"):

| Secret Name | Where to get it | What it does |
|---|---|---|
| `NGROK_TOKEN` | https://dashboard.ngrok.com/get-started/your-authtoken | Creates a public tunnel to your API |
| `HF_TOKEN` | https://huggingface.co/settings/tokens | Downloads model weights from HuggingFace |

> **Tip:** Toggle "Notebook access" ON for each secret after adding it.

---

## 📋 How to use

1. Select **Runtime → Change runtime type → L4 GPU**
2. Add your secrets (see above)
3. Run **Cell 1** — installs dependencies and downloads models (~50 GB, takes ~10-15 min)
4. Run **Cell 2** — loads model and starts the FastAPI + ngrok server
5. Copy the **ngrok URL** printed in the output
6. Send POST requests to `<ngrok_url>/generate` to generate videos

### Example request (Python)
```python
import requests

URL = "https://<your-ngrok-url>/generate"

resp = requests.post(URL, json={
    "prompt": "A golden retriever playing in ocean waves at sunset, slow motion, cinematic",
    "seed": 42,
    "height": 512,
    "width": 768,
    "num_frames": 97
}, timeout=600)

with open("output.mp4", "wb") as f:
    f.write(resp.content)
print("Video saved!")
```

### Example request (curl)
```bash
curl -X POST <ngrok_url>/generate \
  -H "Content-Type: application/json" \
  -d '{"prompt": "A cat sitting on a windowsill watching rain", "seed": 42}' \
  --output output.mp4
```

---

## 📝 Notes
- **Model:** LTX-2.3 22B Distilled + Spatial Upscaler x2 + Gemma 3 12B (Q4)
- **Pipeline:** DistilledPipeline (8 steps stage 1, 4 steps stage 2 — fastest)
- **Quantization:** FP8 cast (fits in 22 GB VRAM)
- **Output:** MP4 video with synchronized audio
- **Default resolution:** 512×768 → upscaled to 1024×1536 (two-stage)
- **Default frames:** 97 (~4 seconds at 24 fps)

In [1]:
# =============================================================================
# CELL 1 — Install Dependencies + Download Models
# =============================================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Install LTX-2 from source ---
print("🔧 Installing LTX-2...")
!git clone --depth 1 https://github.com/Lightricks/LTX-2.git /content/LTX-2 2>/dev/null || echo "Already cloned"
!pip install -q -e /content/LTX-2/packages/ltx-core -e /content/LTX-2/packages/ltx-pipelines 2>&1 | tail -5

# --- Install server dependencies ---
!pip install -q fastapi uvicorn pyngrok huggingface_hub hf_transfer 2>&1 | tail -3

# --- Download model weights ---
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from google.colab import userdata
from huggingface_hub import hf_hub_download, snapshot_download

HF_TOKEN = userdata.get("HF_TOKEN")
MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("\n📦 Downloading LTX-2.3 distilled checkpoint...")
distilled_ckpt = hf_hub_download(
    repo_id="Lightricks/LTX-2.3",
    filename="ltx-2.3-22b-distilled.safetensors",
    local_dir=MODEL_DIR,
    token=HF_TOKEN,
)

print("📦 Downloading spatial upscaler (x2)...")
upsampler_path = hf_hub_download(
    repo_id="Lightricks/LTX-2.3",
    filename="ltx-2.3-spatial-upscaler-x2-1.0.safetensors",
    local_dir=MODEL_DIR,
    token=HF_TOKEN,
)

print("📦 Downloading Gemma 3 text encoder (Q4)...")
gemma_root = snapshot_download(
    repo_id="google/gemma-3-12b-it-qat-q4_0-unquantized",
    local_dir=os.path.join(MODEL_DIR, "gemma-3-12b-it-qat-q4_0-unquantized"),
    token=HF_TOKEN,
)

print(f"\n\u2705 All models downloaded!")
print(f"  Distilled checkpoint: {distilled_ckpt}")
print(f"  Spatial upsampler:    {upsampler_path}")
print(f"  Gemma root:           {gemma_root}")

🔧 Installing LTX-2...
Already cloned

📦 Downloading LTX-2.3 distilled checkpoint...
📦 Downloading spatial upscaler (x2)...
📦 Downloading Gemma 3 text encoder (Q4)...


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]


✅ All models downloaded!
  Distilled checkpoint: /content/models/ltx-2.3-22b-distilled.safetensors
  Spatial upsampler:    /content/models/ltx-2.3-spatial-upscaler-x2-1.0.safetensors
  Gemma root:           /content/models/gemma-3-12b-it-qat-q4_0-unquantized


In [2]:
# Path to the file
file_path = '/content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py'

# Read the file content
with open(file_path, 'r') as file:
    lines = file.readlines()

# Find and replace the specific line
for i in range(len(lines)):
    if 'base = config.rope_local_base_freq' in lines[i]:
        lines[i] = lines[i].replace('base = config.rope_local_base_freq', 'base = getattr(config, "rope_local_base_freq", 10000.0)')

# Write the modified content back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)

print("File updated successfully!")

File updated successfully!


In [3]:
!sed -i '162s/base = config.rope_local_base_freq/base = getattr(config, "rope_local_base_freq", 10000.0)/' /content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py

In [4]:
# Path to the file
file_path = '/content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py'

# Read the file content
with open(file_path, 'r') as file:
    lines = file.readlines()

# Find and replace the specific line
for i in range(len(lines)):
    if 'config.rope_scaling["rope_type"]' in lines[i]:
        lines[i] = lines[i].replace('config.rope_scaling["rope_type"]', 'config.rope_scaling["type"]')

# Write the modified content back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)

print("File updated successfully!")

File updated successfully!


In [8]:
# Path to the file
file_path = '/content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py'

# Read the file content
with open(file_path, 'r') as file:
    lines = file.readlines()

# Find the line with the access and replace it with fallback logic
for i in range(len(lines)):
    if 'ROPE_INIT_FUNCTIONS[config.rope_scaling' in lines[i]:
        lines[i] = '    rope_type = config.rope_scaling.get("rope_type") or config.rope_scaling["type"]\n'
        lines.insert(i+1, '    inv_freqs, _ = ROPE_INIT_FUNCTIONS[rope_type](config)\n')
        break  # assuming only one such line

# Write the modified content back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)

print("File updated successfully!")

File updated successfully!


In [4]:
# Path to the file
file_path = '/content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py'

# Read the file content
with open(file_path, 'r') as file:
    lines = file.readlines()

# Find the line with the rope_type assignment and replace it
for i in range(len(lines)):
    if 'rope_type = config.rope_scaling.get("rope_type") or config.rope_scaling["type"]' in lines[i]:
        lines[i] = '    rope_type = (config.rope_scaling or {}).get("rope_type") or (config.rope_scaling or {}).get("type") or "linear"\n'
        break  # assuming only one such line

# Write the modified content back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)

print("File updated successfully!")

File updated successfully!


In [7]:
# Path to the file
file_path = '/content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py'

# Read the file content
with open(file_path, 'r') as file:
    lines = file.readlines()

# Find the line with 'rope_type ='
for i in range(len(lines)):
    if 'rope_type =' in lines[i]:
        lines[i] = '    rope_type = (config.rope_scaling or {}).get("rope_type") or (config.rope_scaling or {}).get("type") or "linear"\n'
        break

# Write the modified content back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)

print("File updated successfully!")

File updated successfully!


In [10]:
# Path to the file
file_path = '/content/LTX-2/packages/ltx-core/src/ltx_core/text_encoders/gemma/encoders/encoder_configurator.py'

# Read the file content
with open(file_path, 'r') as file:
    lines = file.readlines()

# Find the line containing 'rope_type' and replace it with the safe version
for i in range(len(lines)):
    if 'rope_type' in lines[i] and '=' in lines[i]:
        lines[i] = '    rope_type = (config.rope_scaling or {}).get("rope_type") or (config.rope_scaling or {}).get("type") or "linear"\n'
        break  # assuming only one such assignment

# Write the modified content back to the file
with open(file_path, 'w') as file:
    file.writelines(lines)

print("File updated successfully!")

File updated successfully!


In [11]:
# =============================================================================
# CELL 2 — Load Model + Start FastAPI Server + ngrok Tunnel
# =============================================================================

import os, time, threading, tempfile, uuid, logging, subprocess, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Re-install LTX-2 packages if lost after runtime restart
try:
    import ltx_core
except ModuleNotFoundError:
    print("⚠️ LTX-2 packages not found (runtime restarted?). Re-installing...")
    # Also re-apply the circular import patch
    _loader_init = "/content/LTX-2/packages/ltx-core/src/ltx_core/loader/__init__.py"
    if os.path.exists(_loader_init):
        with open(_loader_init) as f:
            _txt = f.read()
        if "from ltx_core.loader.fuse_loras import apply_loras" in _txt:
            with open(_loader_init, "w") as f:
                f.write(_txt.replace("from ltx_core.loader.fuse_loras import apply_loras\n", ""))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "-e", "/content/LTX-2/packages/ltx-core",
        "-e", "/content/LTX-2/packages/ltx-pipelines"])
    print("✅ Re-installed successfully")

import torch
import uvicorn
from fastapi import FastAPI
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel, Field
from pyngrok import ngrok
from google.colab import userdata

from ltx_core.quantization import QuantizationPolicy
from ltx_core.model.video_vae import TilingConfig, get_video_chunks_number
from ltx_pipelines.distilled import DistilledPipeline
from ltx_pipelines.utils.media_io import encode_video

logging.getLogger().setLevel(logging.INFO)

# ── Config ──
MODEL_DIR = "/content/models"
DISTILLED_CKPT = os.path.join(MODEL_DIR, "ltx-2.3-22b-distilled.safetensors")
UPSAMPLER_PATH = os.path.join(MODEL_DIR, "ltx-2.3-spatial-upscaler-x2-1.0.safetensors")
GEMMA_ROOT = os.path.join(MODEL_DIR, "gemma-3-12b-it-qat-q4_0-unquantized")
OUTPUT_DIR = "/content/outputs"
API_PORT = 8081 # Changed port from 8080 to 8000

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 1. Load the pipeline with FP8 quantization ──
print("🔄 Loading LTX-2.3 DistilledPipeline with FP8 quantization...")
print("   (This takes a few minutes on first load)")

pipeline = DistilledPipeline(
    distilled_checkpoint_path=DISTILLED_CKPT,
    spatial_upsampler_path=UPSAMPLER_PATH,
    gemma_root=GEMMA_ROOT,
    loras=[],
    quantization=QuantizationPolicy.fp8_cast(),
)
print("✅ Pipeline loaded!")

# ── 2. FastAPI app ──
app = FastAPI(title="LTX-2 Video Generation Server")

# Lock to prevent concurrent generation (single GPU)
generation_lock = threading.Lock()


class GenerateRequest(BaseModel):
    prompt: str = Field(..., description="Text prompt describing the video to generate")
    seed: int = Field(default=42, description="Random seed for reproducibility")
    height: int = Field(default=512, description="Video height (will be 2x upscaled). Must be divisible by 32")
    width: int = Field(default=768, description="Video width (will be 2x upscaled). Must be divisible by 32")
    num_frames: int = Field(default=97, description="Number of frames. Must be 8*k+1 (e.g. 25, 49, 97, 121)")
    frame_rate: float = Field(default=24.0, description="Frame rate (fps)")


@app.get("/health")
async def health():
    return {"status": "ok", "model": "LTX-2.3-22B-Distilled", "quantization": "fp8-cast"}


@app.post("/generate")
async def generate(req: GenerateRequest):
    if not generation_lock.acquire(blocking=False):
        return JSONResponse(
            status_code=503,
            content={"error": "A generation is already in progress. Please wait and retry."},
        )
    try:
        video_id = str(uuid.uuid4())[:8]
        output_path = os.path.join(OUTPUT_DIR, f"{video_id}.mp4")

        print(f"\n🎥 Generating video {video_id}...")
        print(f"   Prompt: {req.prompt[:80]}{'...' if len(req.prompt) > 80 else ''}")
        print(f"   Resolution: {req.width}x{req.height} -> {req.width*2}x{req.height*2} (upscaled)")
        print(f"   Frames: {req.num_frames} @ {req.frame_rate} fps")

        start_time = time.time()

        with torch.inference_mode():
            tiling_config = TilingConfig.default()
            video_chunks_number = get_video_chunks_number(req.num_frames, tiling_config)

            video, audio = pipeline(
                prompt=req.prompt,
                seed=req.seed,
                height=req.height * 2,
                width=req.width * 2,
                num_frames=req.num_frames,
                frame_rate=req.frame_rate,
                images=[],
                tiling_config=tiling_config,
            )

            encode_video(
                video=video,
                fps=req.frame_rate,
                audio=audio,
                output_path=output_path,
                video_chunks_number=video_chunks_number,
            )

        elapsed = time.time() - start_time
        print(f"✅ Video generated in {elapsed:.1f}s -> {output_path}")

        return FileResponse(
            output_path,
            media_type="video/mp4",
            filename=f"ltx2_{video_id}.mp4",
            headers={"X-Generation-Time": f"{elapsed:.1f}s"},
        )
    except Exception as e:
        logging.exception("Generation failed")
        return JSONResponse(status_code=500, content={"error": str(e)})
    finally:
        generation_lock.release()


# ── 3. Start ngrok tunnel ──
NGROK_AUTH_TOKEN = userdata.get("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
tunnel = ngrok.connect(API_PORT)
public_url = tunnel.public_url

# ── 4. Run FastAPI in background thread ──
server_thread = threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": API_PORT, "log_level": "warning"},
    daemon=True,
)
server_thread.start()
time.sleep(2)

print("\n" + "=" * 65)
print(f"🌐 PUBLIC API URL: {public_url}")
print(f"🏠 LOCAL API URL:  http://127.0.0.1:{API_PORT}")
print(f"=" * 65)
print(f"\n  GET  {public_url}/health")
print(f"  POST {public_url}/generate")
print(f"  GET  {public_url}/docs   (interactive Swagger UI)")
print(f"\n✅ Server is running! Send requests to generate videos.")
print(f"\nExample:")
print(f'''  curl -X POST {public_url}/generate \n    -H "Content-Type: application/json" \n    -d '{{"prompt": "A cat watching rain from a window", "seed": 42}}' \n    --output video.mp4''')

🔄 Loading LTX-2.3 DistilledPipeline with FP8 quantization...
   (This takes a few minutes on first load)
✅ Pipeline loaded!


INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok
INFO:pyngrok.ngrok:Opening tunnel named: http-8081-60aad715-47ea-4956-ad7b-60cad25b4d61
INFO:pyngrok.process.ngrok:t=2026-03-07T14:59:45+0000 lvl=info msg=start pg=/api/tunnels id=6fe7c305c73c789c
INFO:pyngrok.process.ngrok:t=2026-03-07T14:59:46+0000 lvl=info msg="started tunnel" obj=tunnels name=http-8081-60aad715-47ea-4956-ad7b-60cad25b4d61 addr=http://localhost:8081 url=https://mildred-unspeckled-khadijah.ngrok-free.dev
INFO:pyngrok.process.ngrok:t=2026-03-07T14:59:46+0000 lvl=info msg=end pg=/api/tunnels id=6fe7c305c73c789c status=201 dur=462.348615ms
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8081): address already in use



🌐 PUBLIC API URL: https://mildred-unspeckled-khadijah.ngrok-free.dev
🏠 LOCAL API URL:  http://127.0.0.1:8081

  GET  https://mildred-unspeckled-khadijah.ngrok-free.dev/health
  POST https://mildred-unspeckled-khadijah.ngrok-free.dev/generate
  GET  https://mildred-unspeckled-khadijah.ngrok-free.dev/docs   (interactive Swagger UI)

✅ Server is running! Send requests to generate videos.

Example:
  curl -X POST https://mildred-unspeckled-khadijah.ngrok-free.dev/generate 
    -H "Content-Type: application/json" 
    -d '{"prompt": "A cat watching rain from a window", "seed": 42}' 
    --output video.mp4


In [12]:
# =============================================================================
# CELL 3 — Generate a Video (Quick Test)
# =============================================================================

import requests
from IPython.display import Video, display

API_URL = "http://127.0.0.1:8081/generate"

print("🎥 Generating test video...")
resp = requests.post(API_URL, json={
    "prompt": (
        "A golden retriever puppy runs along a sandy beach at sunset. "
        "Waves crash gently in the background, spraying mist into warm golden light. "
        "The puppy's fur glows in the low sun as it splashes through shallow water. "
        "Camera follows at a low angle, tracking the dog in slow motion. "
        "Cinematic, warm color grading, shallow depth of field."
    ),
    "seed": 42,
    "height": 512,
    "width": 768,
    "num_frames": 97,
    "frame_rate": 24.0,
}, timeout=600)

if resp.status_code == 200:
    output_file = "/content/outputs/test_video.mp4"
    with open(output_file, "wb") as f:
        f.write(resp.content)
    gen_time = resp.headers.get("X-Generation-Time", "unknown")
    print(f"✅ Video saved to {output_file} (generation time: {gen_time})")
    display(Video(output_file, embed=True, width=640))
else:
    print(f"❌ Error: {resp.status_code} - {resp.text}")

ERROR:root:Generation failed
Traceback (most recent call last):
  File "/tmp/ipykernel_30823/971840918.py", line 107, in generate
    video, audio = pipeline(
                   ^^^^^^^^^
  File "/content/LTX-2/packages/ltx-pipelines/src/ltx_pipelines/distilled.py", line 95, in __call__
    (ctx_p,) = encode_prompts(
               ^^^^^^^^^^^^^^^
  File "/content/LTX-2/packages/ltx-pipelines/src/ltx_pipelines/utils/helpers.py", line 71, in encode_prompts
    text_encoder = model_ledger.text_encoder()
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/LTX-2/packages/ltx-pipelines/src/ltx_pipelines/utils/model_ledger.py", line 262, in text_encoder
    return self.text_encoder_builder.build(device=self._target_device(), dtype=self.dtype).to(self.device).eval()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/LTX-2/packages/ltx-core/src/ltx_core/loader/single_gpu_model_builder.py", line 89, in build
    meta_model = s

🎥 Generating test video...

🎥 Generating video ed4a7430...
   Prompt: A golden retriever puppy runs along a sandy beach at sunset. Waves crash gently ...
   Resolution: 768x512 -> 1536x1024 (upscaled)
   Frames: 97 @ 24.0 fps
❌ Error: 500 - {"error":"'type'"}


In [ ]:
# =============================================================================
# CELL 4 — Image-to-Video Generation (Optional)
# =============================================================================
# Upload an image to Colab, then use it as conditioning for video generation.
#
# To use: upload an image via the Colab file browser, then set IMAGE_PATH below.

import requests, base64, json
from IPython.display import Video, display
from google.colab import files

# Upload an image
print("📁 Upload an image to use as conditioning:")
uploaded = files.upload()

if uploaded:
    IMAGE_PATH = "/content/" + list(uploaded.keys())[0]
    print(f"\n🖼️ Using image: {IMAGE_PATH}")
    print("🎥 Generating image-conditioned video...")

    # For image-to-video, we need to use the CLI directly since the
    # DistilledPipeline accepts images as ImageConditioningInput tuples.
    # Using subprocess to call the pipeline with image conditioning:
    import subprocess
    result = subprocess.run([
        "python", "-m", "ltx_pipelines.distilled",
        "--distilled-checkpoint-path", "/content/models/ltx-2.3-22b-distilled.safetensors",
        "--spatial-upsampler-path", "/content/models/ltx-2.3-spatial-upscaler-x2-1.0.safetensors",
        "--gemma-root", "/content/models/gemma-3-12b-it-qat-q4_0-unquantized",
        "--quantization", "fp8-cast",
        "--prompt", "A cinematic video inspired by the input image, smooth camera motion, high quality",
        "--image", IMAGE_PATH, "0", "1.0",
        "--output-path", "/content/outputs/i2v_output.mp4",
        "--seed", "42",
        "--num-frames", "97",
    ], capture_output=True, text=True,
       env={**os.environ, "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})

    print(result.stdout[-500:] if result.stdout else "")
    if result.returncode == 0:
        print("✅ Image-to-video complete!")
        display(Video("/content/outputs/i2v_output.mp4", embed=True, width=640))
    else:
        print(f"❌ Error: {result.stderr[-500:]}")
else:
    print("No image uploaded. Skipping.")